In [43]:
import os
import numpy as np
import pandas as pd

Set display options to inspect all columns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

Load the dataset

In [3]:
data_path = "../port_harcourt_air_quality_dataset/air_quality_historical.csv"
df_raw = pd.read_csv(data_path)
print(f"Dataset successfully loaded! Total rows: {df_raw.shape[0]}, Total columns: {df_raw.shape[1]}")
df_raw.head()

Dataset successfully loaded! Total rows: 1298, Total columns: 12


,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,30.969565,20.339130,288.608696,5.482609,3.786957,57.347826,0.361304,0.000,1.656522,NaN,NaN
4,2022-08-05,33.520833,22.720833,345.458333,5.912500,4.770833,62.500000,0.346250,0.375,1.375000,64.826087,38.434783


Inspect Column Headers

In [ ]:
#Inspect Column Headers for data consistency
df_raw.columns

Index(['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi'], dtype='str')

In [16]:
#Inspect data types and check for missing values
print("====Data Info===")
df_raw.info()


#Missing values audit
print("=====Missing Values Audit =======")
missing_summary = pd.DataFrame({
    "Missing Count": df_raw.isnull().sum(),
    "Missing Percentage (%)": (df_raw.isnull().sum() / len(df_raw)) * 100
})

missing_summary.sort_values(by="Missing Count", ascending=False)

====Data Info===
<class 'pandas.DataFrame'>
RangeIndex: 1298 entries, 0 to 1297
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1298 non-null   str    
 1   pm10                   1295 non-null   float64
 2   pm2_5                  1295 non-null   float64
 3   carbon_monoxide        1295 non-null   float64
 4   nitrogen_dioxide       1295 non-null   float64
 5   sulphur_dioxide        1295 non-null   float64
 6   ozone                  1295 non-null   float64
 7   aerosol_optical_depth  1295 non-null   float64
 8   dust                   1295 non-null   float64
 9   uv_index               1295 non-null   float64
 10  us_aqi                 1294 non-null   float64
 11  european_aqi           1294 non-null   float64
dtypes: float64(11), str(1)
memory usage: 134.5 KB
=====Missing Values Audit =======


,Missing Count,Missing Percentage (%)
european_aqi,4,0.308166
us_aqi,4,0.308166
pm10,3,0.231125
pm2_5,3,0.231125
aerosol_optical_depth,3,0.231125
carbon_monoxide,3,0.231125
nitrogen_dioxide,3,0.231125
sulphur_dioxide,3,0.231125
uv_index,3,0.231125
ozone,3,0.231125


Handling the missing data

In [17]:
# Display only the rows that contain at least one missing value
df_raw[df_raw.isnull().any(axis=1)]

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,30.969565,20.33913,288.608696,5.482609,3.786957,57.347826,0.361304,0.0,1.656522,NaN,NaN


The first 3 rows (2022-08-01, 2022-08-02, 2022-08-03): Completely empty across all pollutants. This is an initialization artifact—the satellite/sensor feed only began recording on August 4, 2022.
The 4th row (2022-08-04): The pollutants (pm2_5, pm10, co, etc.) were captured, but the calculated indices (us_aqi, european_aqi) were missing on the first active day.

Recommended: In data engineering, when missingness is < 0.5% and localized at the startup boundary of a sensor stream, dropping them is the most statistically sound method. You retain 1,296 clean, real-world rows without introducing artificial bias into the ML model.

In [18]:
# 1. Convert 'date' to datetime format for proper time-series indexing
df_clean = df_raw.copy()
df_clean['date'] = pd.to_datetime(df_clean['date'])
# 2. Drop the 4 incomplete initialization rows
df_clean = df_clean.dropna().reset_index(drop=True)

In [27]:
print("====Clean Dataset Check===")
clean_check = pd.DataFrame({
    "dirty_data_count": [len(df_raw)],
    "clean_data_count": [len(df_clean)],
    "Total_rows_dropped": [((len(df_raw) - len(df_clean)) / len(df_raw)) * 100]
}) 

clean_check

====Clean Dataset Check===


,dirty_data_count,clean_data_count,Total_rows_dropped
0,1298,1294,0.308166


In [35]:
# Statistical profile: Mean, Std, Min, 25%, Median, 75%, Max, and Skewness
environmental_cols = [
    'pm2_5', 'pm10', 'dust', 'aerosol_optical_depth',
    'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide',
    'ozone', 'uv_index', 'us_aqi', 'european_aqi'
]
stats_df = df_clean[pollutant_cols].describe().T
stats_df['skewness'] = df_clean[pollutant_cols].skew()
stats_df['iqr'] = stats_df['75%'] - stats_df['25%']
print("=== STATISTICAL HEALTH PROFILE ===")
stats_df[['mean', 'std', 'min', '50%', 'max', 'skewness']]

=== STATISTICAL HEALTH PROFILE ===


,mean,std,min,50%,max,skewness
pm2_5,29.157622,15.739118,0.741667,25.554167,167.279167,3.840004
pm10,43.212725,27.338791,1.054167,35.570833,299.300000,3.732441
carbon_monoxide,449.078310,114.830518,59.208333,429.520833,1111.625000,1.225895
nitrogen_dioxide,7.343917,2.977488,0.000000,6.554167,24.295833,1.735152
sulphur_dioxide,3.004286,2.163867,0.237500,2.137500,14.400000,1.868919
ozone,55.555062,18.274610,26.708333,50.250000,129.916667,1.035906
us_aqi,85.043174,25.315992,15.458333,78.395833,203.000000,1.775903
european_aqi,57.084185,17.873826,13.666667,58.375000,162.625000,1.415317


In [36]:
print("==========================================================================")
print("             NIGER DELTA AIR QUALITY: AUTOMATED COLUMN AUDIT              ")
print("==========================================================================")

for col in environmental_cols:
    series = df_clean[col]
    
    # 1. Basic statistics
    col_min = series.min()
    col_max = series.max()
    col_mean = series.mean()
    col_median = series.median()
    
    # 2. Outlier Detection using the Interquartile Range (IQR) rule
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + (1.5 * iqr)
    outlier_count = (series > upper_bound).sum()
    outlier_pct = (outlier_count / len(series)) * 100
    
    # 3. Validation checks
    has_negative = col_min < 0
    status = "⚠️ REVIEW" if has_negative or (outlier_pct > 10) else "✅ HEALTHY"
    
    # Print clean summary per column
    print(f"\n[{status}] Column: '{col}'")
    print(f"   • Range:         {col_min:.2f}  to  {col_max:.2f}")
    print(f"   • Mean / Median: {col_mean:.2f}  |  {col_median:.2f}")
    print(f"   • Outlier Spikes (> {upper_bound:.2f}): {outlier_count} days ({outlier_pct:.1f}%)")
    if has_negative:
        print(f"   • ❌ ERROR: Contains negative physical values! (Min: {col_min})")

print("\n==========================================================================")

             NIGER DELTA AIR QUALITY: AUTOMATED COLUMN AUDIT              

[✅ HEALTHY] Column: 'pm2_5'
   • Range:         0.74  to  167.28
   • Mean / Median: 29.16  |  25.55
   • Outlier Spikes (> 47.99): 88 days (6.8%)

[✅ HEALTHY] Column: 'pm10'
   • Range:         1.05  to  299.30
   • Mean / Median: 43.21  |  35.57
   • Outlier Spikes (> 73.83): 111 days (8.6%)

[⚠️ REVIEW] Column: 'dust'
   • Range:         0.00  to  313.21
   • Mean / Median: 14.23  |  4.21
   • Outlier Spikes (> 37.66): 143 days (11.1%)

[✅ HEALTHY] Column: 'aerosol_optical_depth'
   • Range:         0.04  to  1.69
   • Mean / Median: 0.49  |  0.46
   • Outlier Spikes (> 0.99): 32 days (2.5%)

[✅ HEALTHY] Column: 'carbon_monoxide'
   • Range:         59.21  to  1111.62
   • Mean / Median: 449.08  |  429.52
   • Outlier Spikes (> 698.40): 42 days (3.2%)

[✅ HEALTHY] Column: 'nitrogen_dioxide'
   • Range:         0.00  to  24.30
   • Mean / Median: 7.34  |  6.55
   • Outlier Spikes (> 13.17): 67 days (5.2%)

[⚠

Feature Engineering (Temporal & Seasonal Enrichment)
Air quality in Port Harcourt is heavily driven by season (Harmattan dry season vs. Atlantic rainy season), we need to extract time features from the date column before saving our clean baseline. We would delimit these data records into year, month, day_of_week, and season: 'Harmattan' (Nov–Feb) vs. 'Wet Season' (Apr–Oct) vs. 'Transition' (March).
aqi_category: EPA classification (Good, Moderate, Unhealthy for Sensitive Groups, Unhealthy, Very Unhealthy)

In [41]:
#1. Temporal Feature Extraction
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['month_name'] = df_clean['date'].dt.strftime('%B')
df_clean['day'] = df_clean['date'].dt.day_name()


#2. Niger Delta Seasonal Classification
def classify_niger_delta_season(month):
    if month in [11, 12, 1, 2]:
        return 'Harmattan / Dry'
    elif month in [4, 5, 6, 7, 8, 9, 10]:
        return 'Rainy / Wet'
    else:
        return 'Transition'

df_clean['season'] = df_clean['month'].apply(classify_niger_delta_season)


# 3. EPA AQI Health Categories
def categorize_aqi(aqi):
    if aqi <= 50:
        return 'Good'
    elif aqi <= 100:
        return 'Moderate'
    elif aqi <= 150:
        return 'Unhealthy for Sensitive Groups'
    elif aqi <= 200:
        return 'Unhealthy'
    else:
        return 'Very Unhealthy'


df_clean['aqi_category'] = df_clean['us_aqi'].apply(categorize_aqi)

# Inspect distribution across seasons and AQI categories
print("=== SEASONAL DISTRIBUTION ===")
print(df_clean['season'].value_counts())
print("\n=== AIR QUALITY HEALTH CATEGORIES (DAYS RECORDED) ===")
print(df_clean['aqi_category'].value_counts())

=== SEASONAL DISTRIBUTION ===
season
Rainy / Wet        730
Harmattan / Dry    471
Transition          93
Name: count, dtype: int64

=== AIR QUALITY HEALTH CATEGORIES (DAYS RECORDED) ===
aqi_category
Moderate                          1066
Unhealthy for Sensitive Groups     173
Unhealthy                           48
Very Unhealthy                       4
Good                                 3
Name: count, dtype: int64


In [42]:
df_clean.head()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi,year,month,month_name,day,season,aqi_category
0,2022-08-05,33.520833,22.720833,345.458333,5.912500,4.770833,62.500,0.346250,0.375000,1.37500,64.826087,38.434783,2022,8,August,Friday,Rainy / Wet,Moderate
1,2022-08-06,30.216667,20.395833,338.000000,7.237500,4.191667,52.875,0.290417,0.125000,0.93750,72.708333,50.833333,2022,8,August,Saturday,Rainy / Wet,Moderate
2,2022-08-07,34.779167,23.616667,388.958333,8.091667,3.666667,53.625,0.340000,0.208333,1.28750,73.625000,52.333333,2022,8,August,Sunday,Rainy / Wet,Moderate
3,2022-08-08,35.175000,24.062500,408.208333,8.441667,4.337500,51.000,0.366667,0.250000,1.14375,75.000000,54.541667,2022,8,August,Monday,Rainy / Wet,Moderate
4,2022-08-09,29.487500,20.112500,357.916667,7.733333,3.620833,49.375,0.333333,0.375000,1.10000,70.375000,46.250000,2022,8,August,Tuesday,Rainy / Wet,Moderate


In [44]:
# 1. Create a dedicated 'data' directory in the project root if it doesn't exist
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

# 2. Save the cleaned and enriched dataset
output_path = os.path.join(output_dir, "clean_port_harcourt_air_quality.csv")
df_clean.to_csv(output_path, index=False)

print("==========================================================================")
print(f"✅ CLEAN BASELINE SAVED SUCCESSFULLY!")
print(f"   • Path:       {os.path.abspath(output_path)}")
print(f"   • Total Rows: {len(df_clean)}")
print(f"   • Features:   {df_clean.shape[1]} columns")
print("==========================================================================")

✅ CLEAN BASELINE SAVED SUCCESSFULLY!
   • Path:       c:\Users\user\Desktop\Projects\RSUT_Exhibtion\data\clean_port_harcourt_air_quality.csv
   • Total Rows: 1294
   • Features:   18 columns
